[Lab README](README.md)

# Lab 5.3: Walk one reservation through the deployed agent (optional)

`5.1_agentcore_deploy.ipynb` ran four smoke tests back to back and printed what
came out. This notebook slows two of them down: an over-limit request that is
refused, then a corrected request that is recorded, then the resulting node read
straight out of the graph, then the log query that ties all of it to one
identifier.

Nothing here deploys or creates anything. It invokes the Runtime that 5.1
already deployed, and reads.

The same `HybridCypherRetriever` from Lab 2 runs inside that Runtime. The agent
has exactly two logical tools: the in-Runtime `search_hotel_knowledge` retrieval
tool, and one `create_reservation_request` command exposed through Gateway. There
is no payment, confirmation, cancellation, or inventory anywhere in this design,
which is why nothing the agent says can imply a booking exists.

**Prerequisite: `AGENT_RUNTIME_ARN`.** 5.1 writes it into the repository-root
`.env` when it launches, and the next cell reads it from there, so if you have
just run 5.1 there is nothing to do. If you are picking this notebook up in a
fresh session, or the Runtime was deployed from another machine, set the value
by hand in that file or export it before starting the kernel. Without it every
live cell below skips.

In [ ]:
import json
import os
import uuid
from datetime import date, timedelta
from pathlib import Path

import boto3
from dotenv import load_dotenv

from workshop.contracts import MAX_GUESTS, OVER_LIMIT_GUESTS
from workshop.graph_setup import HERO_NAME

# The same repo-root walk 5.1 uses. A bare load_dotenv() searches the current
# directory and its parents for a file named .env, and it stops at the first one
# it finds, so a lab-local .env wins and the repo-root one is only reached when
# there is no closer file. Naming the root file explicitly makes the fallback
# certain rather than incidental, which matters here: AGENT_RUNTIME_ARN is
# written to the repo-root .env by 5.1 and read nowhere else.
#
# The marker is setup/run_notebooks.py rather than README.md, because every lab
# folder has a README.md and the walk up would stop at the first one.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "setup" / "run_notebooks.py").is_file():
    if REPO_ROOT == REPO_ROOT.parent:
        raise FileNotFoundError(
            "Repository root not found. Run this notebook from 05-agentcore-deploy/."
        )
    REPO_ROOT = REPO_ROOT.parent
load_dotenv()
load_dotenv(REPO_ROOT / ".env")

RUNTIME_ARN = os.getenv("AGENT_RUNTIME_ARN", "").strip()
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
BEDROCK_READY = boto3.Session().get_credentials() is not None
RUNTIME_READY = bool(RUNTIME_ARN) and BEDROCK_READY

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)

# Relative to today, never a hardcoded date: a fixed future date rots into
# the past and turns a passing stay into a rejected one.
CHECK_IN = (date.today() + timedelta(days=30)).isoformat()
CHECK_OUT = (date.today() + timedelta(days=32)).isoformat()


def invoke_runtime(prompt, request_id):
    client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)
    response = client.invoke_agent_runtime(
        agentRuntimeArn=RUNTIME_ARN,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps(
            {"prompt": prompt, "request_id": request_id}
        ).encode("utf-8"),
        qualifier="DEFAULT",
    )
    return json.loads(response["response"].read())


if not RUNTIME_READY:
    print("No Runtime configured. All live cells below will be skipped.")
    print(f"  Looked for AGENT_RUNTIME_ARN in {REPO_ROOT / '.env'}: "
          f"{'set' if RUNTIME_ARN else 'not set'}")
    print(f"  AWS credentials: {'found' if BEDROCK_READY else 'NOT FOUND'}")
    print("  5.1 writes AGENT_RUNTIME_ARN into that file when it launches.")
    print("This is expected offline.")
else:
    print(f"Runtime configured in {AWS_REGION}.")
    print(f"  {RUNTIME_ARN}")

## 1. One caller-created request ID, reused on retries

The caller creates a single canonical UUID and reuses it for every delivery of the same reservation request. It is the idempotency key and the correlation identifier across Runtime, Gateway, the reservation Lambda, and Neo4j-related work.

In [ ]:
REQUEST_ID = str(uuid.uuid4())
print(f"Caller-created request_id (reuse on retries): {REQUEST_ID}")

## 2. A 15-guest request is rejected with no write

The Neo4j maximum-guests rule caps a reservation request at 10 guests. The command enforces the rule inside the same boundary as the write, so an over-limit request is rejected and nothing is written to the graph.

In [ ]:
if not RUNTIME_READY:
    print("Skipping 15-guest rejection: no Runtime configured.")
else:
    prompt = (
        f"Find {HERO_NAME} and create a reservation request for "
        f"{OVER_LIMIT_GUESTS} guests, check-in {CHECK_IN}, check-out {CHECK_OUT}."
    )
    result = invoke_runtime(prompt, REQUEST_ID)
    print(json.dumps(result, indent=2))

## 3. A corrected request within the limit is recorded

Reusing the same `request_id`, the facilitator submits a request within the 10-guest limit. The command creates one `ReservationRequest` linked to the retrieved hotel by a `FOR_HOTEL` relationship. Re-delivering the identical request returns the existing record without creating a duplicate.

In [ ]:
if not RUNTIME_READY:
    print("Skipping corrected request: no Runtime configured.")
else:
    prompt = (
        f"Find {HERO_NAME} and create a reservation request for "
        f"{MAX_GUESTS} guests, check-in {CHECK_IN}, check-out {CHECK_OUT}."
    )
    result = invoke_runtime(prompt, REQUEST_ID)
    print(json.dumps(result, indent=2))

## 4. Inspect the resulting request in the graph

Anchored on the stable `request_id`, confirm exactly one accepted request linked to exactly one hotel.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph inspection: Neo4j is not configured.")
else:
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(
        os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
    )
    query = (
        "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
        "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
        "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
        "toString(r.created_at) AS created_at"
    )
    try:
        with driver.session(database=os.environ["NEO4J_DATABASE"]) as session:
            for record in session.run(query, rid=REQUEST_ID):
                print(dict(record))
    finally:
        driver.close()

## 5. Correlate AgentCore and CloudWatch by request ID

Every Runtime and Lambda log line records the `request_id` (never prompts, credentials, or connection strings). Use it to trace one reservation across Runtime retrieval, Gateway, the reservation Lambda, and Neo4j-related work. AgentCore Runtime logs land in Amazon CloudWatch under `/aws/bedrock-agentcore/runtimes/`.

In [ ]:
if not RUNTIME_READY:
    print("Skipping log correlation: no Runtime configured.")
else:
    logs = boto3.client("logs", region_name=AWS_REGION)
    print(f"Search CloudWatch Logs Insights for request_id={REQUEST_ID}")
    print("Log group prefix: /aws/bedrock-agentcore/runtimes/")
    print("Example filter pattern:")
    print(f'  fields @timestamp, @message | filter @message like "{REQUEST_ID}" | sort @timestamp asc')
    # A live facilitator can uncomment a filter_log_events call here against the
    # specific runtime and Lambda log groups for their deployment.

## What to look for

Each invocation prints the whole payload. Three of its keys are the model's work
and one is not: `response` is the model's prose, `tools_used` lists the tools it
attempted, `request_id` echoes what you sent, and `command_result` is the
reservation command's own response, computed in the Lambda by a rule that read
the graph. Read `command_result` first. It is the only key no wording can
change, and it is `null` when the command was never reached.

**Expected outcomes**
- 15-guest request: `command_result` carries `status="rejected"` and
  `reason_code="max_guests_exceeded"`, and section 4 finds no graph node.
- Corrected request: `command_result` carries `status="accepted"`, and section 4
  finds one node with a `created_at` timestamp and exactly one `FOR_HOTEL`
  relationship.
- Re-running the corrected cell with the same `REQUEST_ID`: `duplicate=true`,
  the same timestamp, and still one node.

The rejection is the interesting one. No prompt mentions the number 10. The
limit is a `Rule` node in the graph, the command reads it, and the command runs
inside the same boundary as the write. A model that decided to be helpful and
quietly book 10 instead of 15 would still be refused, because the refusal does
not depend on the model having read anything.

**Recovery**
- If a live cell errors, confirm `AGENT_RUNTIME_ARN`, the region, and Bedrock model access, then re-run. Cells are safe to re-run: the command is idempotent by `request_id`.
- To start a clean scenario, restart the kernel so a fresh `REQUEST_ID` is generated.

**When you are done**
- Run `5.2_teardown.ipynb`. The Runtime, its image, and its build project are billing until you do.